# 从零设计关键词搜索：倒排索引、BM25 与工程化

这份 Notebook 不把“关键词搜索”简化成字符串包含判断，而是实现一条可以解释、测试和演进的最小搜索链路：

**需求与查询语法 → 文档/字段/metadata 建模 → 可替换中文分析器 → 带位置的倒排索引 → BM25 打分 → 布尔与短语查询 → 字段权重 → ACL/metadata 前置过滤 → 增量更新 → top-k → 离线评估 → 性能与运维。**

## 学习目标

1. 能解释 postings、TF、DF、IDF、文档长度和 BM25 各变量之间的关系。
2. 能从零实现带词位置信息的倒排索引，并给出逐项 score explanation。
3. 能区分全文字段、精确 keyword 字段和不可参与相关性打分的权限 metadata。
4. 能实现 AND、OR、phrase、多字段权重、ACL、upsert/delete 和稳定 top-k。
5. 能用 Recall@k、MRR、nDCG 与固定回归集判断一次修改到底是改进还是退化。

> **教学边界**：代码仅依赖 Python 标准库，数据常驻内存，便于看清算法。生产环境通常使用 Lucene/Elasticsearch/OpenSearch 等成熟引擎，以获得段合并、WAL、压缩 postings、并发读写、分布式容错和成熟查询解析器。

## 1. 先写搜索合同：搜索什么、允许谁看、怎样算“相关”

实现前先固定合同，否则“搜不到”和“排得不好”会混在一起。

| 维度 | 本教程的选择 | 生产问题 |
|---|---|---|
| 可检索字段 | title、body、tags | 字段是否使用相同 analyzer？ |
| 精确 metadata | tenant、allowed_roles、category、status、version | 哪些必须在打分前过滤？ |
| 查询语法 | OR 默认；显式 AND；双引号 phrase；title:/body:/tags: 字段限定 | 非法语法是报错还是降级？ |
| 排序 | 多字段 BM25 加权 + phrase boost + 稳定 doc_id tie-break | 是否叠加时效、质量、业务规则？ |
| 权限 | tenant 相等、角色有交集、status=active | 缓存键是否包含身份与索引代次？ |
| 更新 | doc_id 幂等 upsert；delete 移除 postings | 删除何时对所有副本和缓存生效？ |

示例：

- **BM25 错误码**：默认 OR，任一子句可召回；
- **BM25 AND 错误码**：两个子句都必须命中；
- **"倒排 索引"**：词位必须连续；
- **title:BM25 AND body:"倒排 索引"**：字段限定；
- category 等过滤条件通过结构化参数传入，不和用户全文字符串拼接，避免解析歧义和过滤注入。

搜索接口还应约定 top_k 上限、超时、分页方式、空查询行为、错误码、可解释模式和结果中的 source/version。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
from typing import Iterable  # 导入本单元所需的依赖。
import heapq  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。

SEARCH_FIELDS = ("title", "body", "tags")  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Document:  # 定义承载本节状态与行为的数据结构。
    doc_id: str  # 执行当前语句以推进本节示例。
    title: str  # 执行当前语句以推进本节示例。
    body: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    allowed_roles: frozenset[str]  # 执行当前语句以推进本节示例。
    category: str  # 执行当前语句以推进本节示例。
    status: str  # 执行当前语句以推进本节示例。
    version: int  # 执行当前语句以推进本节示例。
    tags: tuple[str, ...]  # 执行当前语句以推进本节示例。

documents = [  # 计算并保存当前步骤的中间状态。
    Document(  # 执行当前语句以推进本节示例。
        "search-guide", "BM25 与关键词搜索",  # 执行当前语句以推进本节示例。
        "倒排索引记录词项、文档频率和位置。BM25 适合错误码与精确实体。",  # 执行当前语句以推进本节示例。
        "acme", frozenset({"employee", "support"}), "guide", "active", 2,  # 执行当前语句以推进本节示例。
        ("检索", "BM25"),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
    Document(  # 执行当前语句以推进本节示例。
        "auth-runbook", "登录故障 ERR_AUTH_403",  # 执行当前语句以推进本节示例。
        "出现 ERR_AUTH_403 时，检查 ACL 权限和租户配置。",  # 执行当前语句以推进本节示例。
        "acme", frozenset({"support", "admin"}), "runbook", "active", 3,  # 执行当前语句以推进本节示例。
        ("故障", "权限"),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
    Document(  # 执行当前语句以推进本节示例。
        "beta-secret", "Beta 租户内部价格",  # 执行当前语句以推进本节示例。
        "Beta 专属套餐价格为 999 元，不得跨租户展示。",  # 执行当前语句以推进本节示例。
        "beta", frozenset({"employee"}), "confidential", "active", 1,  # 执行当前语句以推进本节示例。
        ("价格",),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
    Document(  # 执行当前语句以推进本节示例。
        "search-guide-old", "旧版关键词搜索",  # 执行当前语句以推进本节示例。
        "旧版建议仅使用 TF-IDF。",  # 执行当前语句以推进本节示例。
        "acme", frozenset({"employee"}), "guide", "archived", 1,  # 执行当前语句以推进本节示例。
        ("检索", "旧版"),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
    Document(  # 执行当前语句以推进本节示例。
        "zh-analyzer", "中文分词分析器",  # 执行当前语句以推进本节示例。
        "中文关键词搜索需要分词，并保留产品型号 AX-100 和数字 650。",  # 执行当前语句以推进本节示例。
        "acme", frozenset({"employee", "search"}), "guide", "active", 1,  # 执行当前语句以推进本节示例。
        ("中文", "分词"),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
    Document(  # 执行当前语句以推进本节示例。
        "index-maintenance", "索引维护",  # 执行当前语句以推进本节示例。
        "倒排结构负责词项查找，索引还需要增量更新和删除。",  # 执行当前语句以推进本节示例。
        "acme", frozenset({"employee"}), "operations", "active", 1,  # 执行当前语句以推进本节示例。
        ("索引", "增量"),  # 执行当前语句以推进本节示例。
    ),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

assert len({doc.doc_id for doc in documents}) == len(documents)  # 用受控断言验证关键不变量。
print("示例文档数:", len(documents))  # 执行当前语句以推进本节示例。

## 2. 文档、字段与 metadata：不要把所有内容塞进一个 text

字段类型决定索引方式：

- **全文字段** title/body：经过 analyzer，保存 term frequency 与 positions，参与 BM25。
- **多值全文字段** tags：本教程把 tag 连接后分析；真实引擎通常保留 value boundary。
- **keyword/数值字段** tenant、category、status、version：做精确过滤、聚合或排序，不应被中文分词。
- **权限字段** allowed_roles：必须在候选打分前下推。先全库召回再在应用层删除，会浪费 top-k 名额，还可能通过结果数量、缓存或 explain 泄漏信息。
- **身份与版本** doc_id/version：doc_id 是幂等更新键；业务上还常需要 family_id、effective_at、content_hash、source_uri、updated_at。

映射设计是 API 合同。修改 analyzer 后，同一个文本会产生不同 term，通常需要新索引全量重建并通过 alias 原子切换，而不是偷偷原地变更。

## 3. 中文 analyzer 是可替换组件

典型分析链：

1. Unicode 规范化、大小写与全半角处理；
2. 字符过滤（HTML、控制符等）；
3. tokenizer（中文词典、统计分词、子词或字符 n-gram）；
4. token filter（同义词、停用词、词干、去重等）；
5. 保存 term、position、offset、positionLength。

**索引时和查询时必须兼容**。若索引把“倒排索引”切为“倒排/索引”，查询却得到单个“倒排索引”，将无法命中。错误码、产品型号、版本号和金额要尽量整体保留；无脑去标点会把 AX-100 破坏为 AX 与 100。

下面是“领域词典最长匹配 + ASCII 型号整体保留 + 未登录汉字退化为单字”的教学 analyzer。它可被同一接口的 jieba、IK、SmartCN 或业务 tokenizer 替换，但不代表生产级中文分词质量。

In [ ]:
class SimpleChineseAnalyzer:  # 定义承载本节状态与行为的数据结构。
    """教学 analyzer：返回规范化 token；真实系统还应返回 offset/type。"""  # 执行当前语句以推进本节示例。

    def __init__(self, lexicon: Iterable[str]):  # 定义本节可复用的核心函数。
        self.lexicon = sorted(set(lexicon), key=lambda word: (-len(word), word))  # 计算并保存当前步骤的中间状态。

    def analyze(self, text: str) -> list[str]:  # 定义本节可复用的核心函数。
        text = text.lower()  # 计算并保存当前步骤的中间状态。
        tokens: list[str] = []  # 计算并保存当前步骤的中间状态。
        i = 0  # 计算并保存当前步骤的中间状态。
        while i < len(text):  # 在终止条件满足前持续推进状态。
            ch = text[i]  # 计算并保存当前步骤的中间状态。
            if ch.isspace() or ch in "，。！？；：、（）()[]{}“”\"'":  # 按当前条件选择后续控制路径。
                i += 1  # 计算并保存当前步骤的中间状态。
                continue  # 调整当前循环或占位控制流。
            ascii_match = re.match(r"[a-z0-9]+(?:[_-][a-z0-9]+)*", text[i:])  # 计算并保存当前步骤的中间状态。
            if ascii_match:  # 按当前条件选择后续控制路径。
                token = ascii_match.group(0)  # 计算并保存当前步骤的中间状态。
                tokens.append(token)  # 执行当前语句以推进本节示例。
                i += len(token)  # 计算并保存当前步骤的中间状态。
                continue  # 调整当前循环或占位控制流。
            matched = next((word for word in self.lexicon if text.startswith(word.lower(), i)), None)  # 计算并保存当前步骤的中间状态。
            if matched:  # 按当前条件选择后续控制路径。
                tokens.append(matched.lower())  # 执行当前语句以推进本节示例。
                i += len(matched)  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                tokens.append(ch)  # 执行当前语句以推进本节示例。
                i += 1  # 计算并保存当前步骤的中间状态。
        return tokens  # 返回当前分支计算出的结果。

LEXICON = {  # 计算并保存当前步骤的中间状态。
    "关键词", "搜索", "倒排", "索引", "记录", "词项", "文档频率", "位置",  # 执行当前语句以推进本节示例。
    "适合", "错误码", "精确实体", "登录", "故障", "检查", "权限", "租户",  # 执行当前语句以推进本节示例。
    "配置", "中文", "分词", "分析器", "需要", "产品型号", "数字", "结构",  # 执行当前语句以推进本节示例。
    "负责", "查找", "增量", "更新", "删除", "旧版", "建议", "价格",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
analyzer = SimpleChineseAnalyzer(LEXICON)  # 计算并保存当前步骤的中间状态。

samples = ["倒排索引", "ERR_AUTH_403", "产品 AX-100 每晚 650 元"]  # 计算并保存当前步骤的中间状态。
for sample in samples:  # 遍历输入元素以累积或检查结果。
    print(sample, "=>", analyzer.analyze(sample))  # 计算并保存当前步骤的中间状态。

In [ ]:
assert analyzer.analyze("倒排索引") == ["倒排", "索引"]  # 用受控断言验证关键不变量。
assert analyzer.analyze("ERR_AUTH_403") == ["err_auth_403"]  # 用受控断言验证关键不变量。
assert "ax-100" in analyzer.analyze("产品 AX-100")  # 用受控断言验证关键不变量。
assert analyzer.analyze("BM25") == ["bm25"]  # 用受控断言验证关键不变量。

# analyzer 版本应写进索引 manifest 与缓存键；这里用固定字符串模拟。
ANALYZER_VERSION = "simple-zh-v1"  # 计算并保存当前步骤的中间状态。
print("analyzer 回归测试通过，版本:", ANALYZER_VERSION)  # 执行当前语句以推进本节示例。

## 4. 倒排索引：从文档找词，变成从词找文档

正排结构是 doc_id → tokens；倒排结构是 field → term → postings。一个 posting 至少可含：

- doc_id；
- 词频 TF，即该 term 在字段中出现次数；
- positions，用于 phrase/proximity；
- offsets，用于高亮（本教程未实现）。

对词项 t：

$$
TF(t,d)=f(t,d), \qquad DF(t)=|\{d:t\in d\}|.
$$

DF 是“包含该词的文档数”，不是词在全库出现的总次数。常见错误是用 collection frequency 代替 DF。

本实现同时保存正排 token。这样 delete/upsert 能找到旧文档的全部 term 并移除 postings；代价是内存更多。成熟引擎通常写不可变 segment，delete 先打 tombstone，后台 merge 时物理回收。

In [ ]:
class InMemoryBM25Index:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, analyzer: SimpleChineseAnalyzer, k1: float = 1.2, b: float = 0.75):  # 定义本节可复用的核心函数。
        if k1 < 0 or not 0 <= b <= 1:  # 按当前条件选择后续控制路径。
            raise ValueError("要求 k1 >= 0 且 0 <= b <= 1")  # 遇到非法合同立即显式失败。
        self.analyzer = analyzer  # 计算并保存当前步骤的中间状态。
        self.k1 = k1  # 计算并保存当前步骤的中间状态。
        self.b = b  # 计算并保存当前步骤的中间状态。
        self.docs: dict[str, Document] = {}  # 计算并保存当前步骤的中间状态。
        # postings[field][term][doc_id] = tuple(positions) 中文说明：该注释解释本行约束。
        self.postings = {field: defaultdict(dict) for field in SEARCH_FIELDS}  # 计算并保存当前步骤的中间状态。
        self.doc_tokens: dict[str, dict[str, list[str]]] = {}  # 计算并保存当前步骤的中间状态。
        self.generation = 0  # 计算并保存当前步骤的中间状态。

    @staticmethod  # 为下方定义附加声明式配置。
    def _field_text(doc: Document, field: str) -> str:  # 定义本节可复用的核心函数。
        value = getattr(doc, field)  # 计算并保存当前步骤的中间状态。
        return " ".join(value) if isinstance(value, tuple) else value  # 返回当前分支计算出的结果。

    def _remove_without_generation(self, doc_id: str) -> bool:  # 定义本节可复用的核心函数。
        if doc_id not in self.docs:  # 按当前条件选择后续控制路径。
            return False  # 返回当前分支计算出的结果。
        for field, tokens in self.doc_tokens[doc_id].items():  # 遍历输入元素以累积或检查结果。
            for term in set(tokens):  # 遍历输入元素以累积或检查结果。
                posting = self.postings[field].get(term)  # 计算并保存当前步骤的中间状态。
                if posting is not None:  # 按当前条件选择后续控制路径。
                    posting.pop(doc_id, None)  # 执行当前语句以推进本节示例。
                    if not posting:  # 按当前条件选择后续控制路径。
                        del self.postings[field][term]  # 执行当前语句以推进本节示例。
        del self.doc_tokens[doc_id]  # 执行当前语句以推进本节示例。
        del self.docs[doc_id]  # 执行当前语句以推进本节示例。
        return True  # 返回当前分支计算出的结果。

    def upsert(self, doc: Document) -> None:  # 定义本节可复用的核心函数。
        self._remove_without_generation(doc.doc_id)  # 执行当前语句以推进本节示例。
        tokens_by_field: dict[str, list[str]] = {}  # 计算并保存当前步骤的中间状态。
        for field in SEARCH_FIELDS:  # 遍历输入元素以累积或检查结果。
            tokens = self.analyzer.analyze(self._field_text(doc, field))  # 计算并保存当前步骤的中间状态。
            tokens_by_field[field] = tokens  # 计算并保存当前步骤的中间状态。
            positions_by_term = defaultdict(list)  # 计算并保存当前步骤的中间状态。
            for position, term in enumerate(tokens):  # 遍历输入元素以累积或检查结果。
                positions_by_term[term].append(position)  # 执行当前语句以推进本节示例。
            for term, positions in positions_by_term.items():  # 遍历输入元素以累积或检查结果。
                self.postings[field][term][doc.doc_id] = tuple(positions)  # 计算并保存当前步骤的中间状态。
        self.docs[doc.doc_id] = doc  # 计算并保存当前步骤的中间状态。
        self.doc_tokens[doc.doc_id] = tokens_by_field  # 计算并保存当前步骤的中间状态。
        self.generation += 1  # 计算并保存当前步骤的中间状态。

    def delete(self, doc_id: str) -> bool:  # 定义本节可复用的核心函数。
        removed = self._remove_without_generation(doc_id)  # 计算并保存当前步骤的中间状态。
        if removed:  # 按当前条件选择后续控制路径。
            self.generation += 1  # 计算并保存当前步骤的中间状态。
        return removed  # 返回当前分支计算出的结果。

    def avg_field_length(self, field: str, universe: set[str]) -> float:  # 定义本节可复用的核心函数。
        if not universe:  # 按当前条件选择后续控制路径。
            return 0.0  # 返回当前分支计算出的结果。
        return sum(len(self.doc_tokens[doc_id][field]) for doc_id in universe) / len(universe)  # 返回当前分支计算出的结果。

    def doc_frequency(self, field: str, term: str, universe: set[str]) -> int:  # 定义本节可复用的核心函数。
        posting_ids = self.postings[field].get(term, {})  # 计算并保存当前步骤的中间状态。
        return sum(doc_id in universe for doc_id in posting_ids)  # 返回当前分支计算出的结果。

index = InMemoryBM25Index(analyzer)  # 计算并保存当前步骤的中间状态。
for document in documents:  # 遍历输入元素以累积或检查结果。
    index.upsert(document)  # 执行当前语句以推进本节示例。

print("index generation:", index.generation)  # 执行当前语句以推进本节示例。
print("body/倒排 postings:", dict(index.postings["body"]["倒排"]))  # 执行当前语句以推进本节示例。
assert index.postings["body"]["倒排"]["search-guide"] == (0,)  # 用受控断言验证关键不变量。

## 5. BM25：不是“TF × IDF”那么简单

本教程使用 Robertson/Sparck Jones 风格的平滑 IDF：

$$
IDF(t)=\log\left(1+\frac{N-DF(t)+0.5}{DF(t)+0.5}\right).
$$

单字段贡献为：

$$
score(t,d)=IDF(t)\cdot
\frac{TF(t,d)(k_1+1)}
{TF(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}.
$$

变量含义：

- N：本次统计空间中的可见文档数；
- DF(t)：可见文档里包含 t 的文档数；
- TF(t,d)：t 在 d 的该字段出现次数；
- |d| / avgdl：字段长度相对平均长度；
- k1：TF 饱和速度，越大越容许重复词继续加分；
- b：长度归一化强度，0 表示不做长度归一化，1 表示完全使用长度比。

BM25 分数不是概率，不应把 7.3 解读为“73% 相关”。跨查询、跨索引、跨 analyzer 的绝对分数一般不可直接比较。这里在**权限过滤后的可见集合**计算统计量，避免跨租户统计泄漏；大规模系统更常用物理 tenant 分片或预计算的安全统计。

In [ ]:
FIELD_WEIGHTS = {"title": 2.2, "body": 1.0, "tags": 1.4}  # 计算并保存当前步骤的中间状态。

def bm25_term_contribution(  # 定义本节可复用的核心函数。
    idx: InMemoryBM25Index,  # 执行当前语句以推进本节示例。
    doc_id: str,  # 执行当前语句以推进本节示例。
    field: str,  # 执行当前语句以推进本节示例。
    term: str,  # 执行当前语句以推进本节示例。
    universe: set[str],  # 执行当前语句以推进本节示例。
) -> dict:  # 执行当前语句以推进本节示例。
    tf = len(idx.postings[field].get(term, {}).get(doc_id, ()))  # 计算并保存当前步骤的中间状态。
    n = len(universe)  # 计算并保存当前步骤的中间状态。
    df = idx.doc_frequency(field, term, universe)  # 计算并保存当前步骤的中间状态。
    avgdl = idx.avg_field_length(field, universe)  # 计算并保存当前步骤的中间状态。
    dl = len(idx.doc_tokens[doc_id][field])  # 计算并保存当前步骤的中间状态。
    idf = math.log(1 + (n - df + 0.5) / (df + 0.5)) if n else 0.0  # 计算并保存当前步骤的中间状态。
    norm = idx.k1 * (1 - idx.b + idx.b * dl / avgdl) if avgdl else idx.k1  # 计算并保存当前步骤的中间状态。
    raw = idf * tf * (idx.k1 + 1) / (tf + norm) if tf else 0.0  # 计算并保存当前步骤的中间状态。
    weighted = raw * FIELD_WEIGHTS[field]  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "term": term, "field": field, "tf": tf, "df": df, "N": n,  # 执行当前语句以推进本节示例。
        "dl": dl, "avgdl": avgdl, "idf": idf,  # 执行当前语句以推进本节示例。
        "field_weight": FIELD_WEIGHTS[field], "contribution": weighted,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

visible_employee = {  # 计算并保存当前步骤的中间状态。
    doc_id for doc_id, doc in index.docs.items()  # 执行当前语句以推进本节示例。
    if doc.tenant == "acme"  # 按当前条件选择后续控制路径。
    and doc.status == "active"  # 计算并保存当前步骤的中间状态。
    and doc.allowed_roles & {"employee"}  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
explanation = bm25_term_contribution(  # 计算并保存当前步骤的中间状态。
    index, "search-guide", "title", "bm25", visible_employee  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
for key, value in explanation.items():  # 遍历输入元素以累积或检查结果。
    print(f"{key:>14}: {value:.4f}" if isinstance(value, float) else f"{key:>14}: {value}")  # 执行当前语句以推进本节示例。

## 6. 查询解析：AND、OR、phrase 与字段限定

查询解析器不是用 split 就能可靠完成的。生产语法还要处理括号、NOT、转义、通配符、最大子句数、超长 phrase、字段白名单和错误位置。本教程只实现一小套明确语法：

- 空格分隔普通 clause；
- AND 出现时，所有 clause 集合取交集，否则取并集；
- 双引号表示 phrase；
- title:、body:、tags: 限定字段；
- 未知字段、空 phrase、只有 AND/OR 的查询报 ValueError；
- metadata filter 不从全文语法推断，而是独立结构化参数。

普通 clause 经 analyzer 后可能产生多个 token，本实现认为“命中其中任一 token”即命中该 clause；phrase 则要求全部 token 在同一字段连续出现。这是教学语义，正式 API 必须把语义写入合同。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class QueryClause:  # 定义承载本节状态与行为的数据结构。
    field: str | None  # 执行当前语句以推进本节示例。
    terms: tuple[str, ...]  # 执行当前语句以推进本节示例。
    is_phrase: bool  # 执行当前语句以推进本节示例。
    raw: str  # 执行当前语句以推进本节示例。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class ParsedQuery:  # 定义承载本节状态与行为的数据结构。
    mode: str  # 执行当前语句以推进本节示例。
    clauses: tuple[QueryClause, ...]  # 执行当前语句以推进本节示例。

TOKEN_PATTERN = re.compile(  # 计算并保存当前步骤的中间状态。
    r'(?:(?P<field>[A-Za-z_][A-Za-z0-9_]*):)?'  # 执行当前语句以推进本节示例。
    r'(?:"(?P<phrase>[^"]*)"|(?P<word>\S+))'  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。

def parse_query(query: str, analyzer: SimpleChineseAnalyzer) -> ParsedQuery:  # 定义本节可复用的核心函数。
    mode = "AND" if re.search(r"\bAND\b", query, flags=re.IGNORECASE) else "OR"  # 计算并保存当前步骤的中间状态。
    clauses: list[QueryClause] = []  # 计算并保存当前步骤的中间状态。
    for match in TOKEN_PATTERN.finditer(query):  # 遍历输入元素以累积或检查结果。
        field = match.group("field")  # 计算并保存当前步骤的中间状态。
        raw = match.group("phrase") if match.group("phrase") is not None else match.group("word")  # 计算并保存当前步骤的中间状态。
        if raw.upper() in {"AND", "OR"} and field is None:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        if field is not None and field not in SEARCH_FIELDS:  # 按当前条件选择后续控制路径。
            raise ValueError(f"未知全文字段: {field}")  # 遇到非法合同立即显式失败。
        terms = tuple(analyzer.analyze(raw))  # 计算并保存当前步骤的中间状态。
        if not terms:  # 按当前条件选择后续控制路径。
            raise ValueError("查询 clause 分析后为空")  # 遇到非法合同立即显式失败。
        clauses.append(QueryClause(field, terms, match.group("phrase") is not None, raw))  # 执行当前语句以推进本节示例。
    if not clauses:  # 按当前条件选择后续控制路径。
        raise ValueError("查询至少需要一个有效 clause")  # 遇到非法合同立即显式失败。
    return ParsedQuery(mode, tuple(clauses))  # 返回当前分支计算出的结果。

for text in ['BM25 AND 错误码', '"倒排 索引"', 'title:BM25 AND body:"倒排 索引"']:  # 遍历输入元素以累积或检查结果。
    print(text, "=>", parse_query(text, analyzer))  # 计算并保存当前步骤的中间状态。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class SearchHit:  # 定义承载本节状态与行为的数据结构。
    doc: Document  # 执行当前语句以推进本节示例。
    score: float  # 执行当前语句以推进本节示例。
    explanation: tuple[dict, ...]  # 执行当前语句以推进本节示例。

def visible_doc_ids(  # 定义本节可复用的核心函数。
    idx: InMemoryBM25Index,  # 执行当前语句以推进本节示例。
    tenant: str,  # 执行当前语句以推进本节示例。
    roles: set[str],  # 执行当前语句以推进本节示例。
    metadata_filter: dict[str, object] | None = None,  # 计算并保存当前步骤的中间状态。
) -> set[str]:  # 执行当前语句以推进本节示例。
    metadata_filter = metadata_filter or {}  # 计算并保存当前步骤的中间状态。
    allowed_keys = {"category", "version"}  # 计算并保存当前步骤的中间状态。
    unknown = set(metadata_filter) - allowed_keys  # 计算并保存当前步骤的中间状态。
    if unknown:  # 按当前条件选择后续控制路径。
        raise ValueError(f"不允许的 filter 字段: {sorted(unknown)}")  # 遇到非法合同立即显式失败。
    result = set()  # 计算并保存当前步骤的中间状态。
    for doc_id, doc in idx.docs.items():  # 遍历输入元素以累积或检查结果。
        if doc.tenant != tenant or doc.status != "active":  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        if not doc.allowed_roles.intersection(roles):  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        if any(getattr(doc, key) != value for key, value in metadata_filter.items()):  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        result.add(doc_id)  # 执行当前语句以推进本节示例。
    return result  # 返回当前分支计算出的结果。

def clause_match_ids(idx: InMemoryBM25Index, clause: QueryClause) -> set[str]:  # 定义本节可复用的核心函数。
    fields = (clause.field,) if clause.field else SEARCH_FIELDS  # 计算并保存当前步骤的中间状态。
    if not clause.is_phrase:  # 按当前条件选择后续控制路径。
        return {  # 返回当前分支计算出的结果。
            doc_id  # 执行当前语句以推进本节示例。
            for field in fields  # 遍历输入元素以累积或检查结果。
            for term in clause.terms  # 遍历输入元素以累积或检查结果。
            for doc_id in idx.postings[field].get(term, {})  # 遍历输入元素以累积或检查结果。
        }  # 执行当前语句以推进本节示例。

    matched = set()  # 计算并保存当前步骤的中间状态。
    for field in fields:  # 遍历输入元素以累积或检查结果。
        first_postings = idx.postings[field].get(clause.terms[0], {})  # 计算并保存当前步骤的中间状态。
        for doc_id, starts in first_postings.items():  # 遍历输入元素以累积或检查结果。
            later_positions = [  # 计算并保存当前步骤的中间状态。
                set(idx.postings[field].get(term, {}).get(doc_id, ()))  # 执行当前语句以推进本节示例。
                for term in clause.terms[1:]  # 遍历输入元素以累积或检查结果。
            ]  # 执行当前语句以推进本节示例。
            if any(all(start + offset in positions  # 按当前条件选择后续控制路径。
                       for offset, positions in enumerate(later_positions, start=1))  # 遍历输入元素以累积或检查结果。
                   for start in starts):  # 遍历输入元素以累积或检查结果。
                matched.add(doc_id)  # 执行当前语句以推进本节示例。
    return matched  # 返回当前分支计算出的结果。

def search(  # 定义本节可复用的核心函数。
    idx: InMemoryBM25Index,  # 执行当前语句以推进本节示例。
    query: str,  # 执行当前语句以推进本节示例。
    tenant: str,  # 执行当前语句以推进本节示例。
    roles: set[str],  # 执行当前语句以推进本节示例。
    top_k: int = 10,  # 计算并保存当前步骤的中间状态。
    metadata_filter: dict[str, object] | None = None,  # 计算并保存当前步骤的中间状态。
    phrase_boost: float = 1.5,  # 计算并保存当前步骤的中间状态。
) -> list[SearchHit]:  # 执行当前语句以推进本节示例。
    if not 1 <= top_k <= 100:  # 按当前条件选择后续控制路径。
        raise ValueError("top_k 必须在 1..100")  # 遇到非法合同立即显式失败。
    parsed = parse_query(query, idx.analyzer)  # 计算并保存当前步骤的中间状态。
    visible = visible_doc_ids(idx, tenant, roles, metadata_filter)  # 计算并保存当前步骤的中间状态。
    match_sets = [clause_match_ids(idx, clause) for clause in parsed.clauses]  # 计算并保存当前步骤的中间状态。
    candidates = (  # 计算并保存当前步骤的中间状态。
        set.intersection(*match_sets) if parsed.mode == "AND"  # 计算并保存当前步骤的中间状态。
        else set.union(*match_sets)  # 执行当前语句以推进本节示例。
    ) & visible  # 执行当前语句以推进本节示例。

    hits = []  # 计算并保存当前步骤的中间状态。
    for doc_id in candidates:  # 遍历输入元素以累积或检查结果。
        details: list[dict] = []  # 计算并保存当前步骤的中间状态。
        for clause in parsed.clauses:  # 遍历输入元素以累积或检查结果。
            fields = (clause.field,) if clause.field else SEARCH_FIELDS  # 计算并保存当前步骤的中间状态。
            for field in fields:  # 遍历输入元素以累积或检查结果。
                for term in clause.terms:  # 遍历输入元素以累积或检查结果。
                    detail = bm25_term_contribution(idx, doc_id, field, term, visible)  # 计算并保存当前步骤的中间状态。
                    if detail["contribution"]:  # 按当前条件选择后续控制路径。
                        details.append(detail)  # 执行当前语句以推进本节示例。
            if clause.is_phrase and doc_id in clause_match_ids(idx, clause):  # 按当前条件选择后续控制路径。
                details.append({  # 执行当前语句以推进本节示例。
                    "term": f'PHRASE({clause.raw})', "field": clause.field or "*",  # 执行当前语句以推进本节示例。
                    "tf": 1, "df": None, "N": len(visible), "dl": None,  # 执行当前语句以推进本节示例。
                    "avgdl": None, "idf": None, "field_weight": phrase_boost,  # 执行当前语句以推进本节示例。
                    "contribution": phrase_boost,  # 执行当前语句以推进本节示例。
                })  # 执行当前语句以推进本节示例。
        score = sum(item["contribution"] for item in details)  # 计算并保存当前步骤的中间状态。
        hits.append(SearchHit(idx.docs[doc_id], score, tuple(details)))  # 执行当前语句以推进本节示例。

    return sorted(hits, key=lambda hit: (-hit.score, hit.doc.doc_id))[:top_k]  # 返回当前分支计算出的结果。

hits = search(index, 'title:BM25 AND body:"倒排 索引"',  # 计算并保存当前步骤的中间状态。
              tenant="acme", roles={"employee"}, top_k=5)  # 计算并保存当前步骤的中间状态。
print([(hit.doc.doc_id, round(hit.score, 4)) for hit in hits])  # 执行当前语句以推进本节示例。
assert [hit.doc.doc_id for hit in hits] == ["search-guide"]  # 用受控断言验证关键不变量。

## 7. 可解释分数与多字段：加权 BM25 不等于严格 BM25F

当前实现对每个字段独立计算 BM25，再乘 title=2.2、body=1.0、tags=1.4 后相加。它简单、可解释，常作为可靠基线：

$$
score(q,d)=\sum_f w_f\sum_{t\in q}BM25_f(t,d).
$$

严格 BM25F 会先按字段长度和字段权重组合归一化 TF，再用跨字段统计量计算一个 term 分数。两者不完全相同。Elasticsearch combined_fields 也说明它基于简化 BM25F 并存在近似。不要把字段 boost 随意调到几十倍；先看 qrels、explain 和字段长度分布。

explain 至少要显示 analyzer token、命中字段、TF、DF、N、dl、avgdl、IDF、字段权重、各项贡献和最终规则 boost。线上可只对受控调试请求开放 explain，因为它更慢，也可能暴露语料统计。

In [ ]:
debug_hit = search(  # 计算并保存当前步骤的中间状态。
    index, "BM25 错误码", tenant="acme",  # 计算并保存当前步骤的中间状态。
    roles={"employee", "support"}, top_k=1  # 计算并保存当前步骤的中间状态。
)[0]  # 执行当前语句以推进本节示例。
print("doc:", debug_hit.doc.doc_id, "total:", round(debug_hit.score, 4))  # 执行当前语句以推进本节示例。
for item in debug_hit.explanation:  # 遍历输入元素以累积或检查结果。
    print(  # 执行当前语句以推进本节示例。
        item["term"], "@", item["field"],  # 执行当前语句以推进本节示例。
        "tf=", item["tf"], "df=", item["df"],  # 计算并保存当前步骤的中间状态。
        "contribution=", round(item["contribution"], 4),  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
assert math.isclose(  # 用受控断言验证关键不变量。
    debug_hit.score,  # 执行当前语句以推进本节示例。
    sum(item["contribution"] for item in debug_hit.explanation),  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。

## 8. 权限、过滤与查询语义的可执行检查

需要分别验证：

1. ACL 不只是结果后处理：不可见 doc_id 根本不进入 candidates 与统计空间；
2. archived 文档无论多相关都不返回；
3. metadata filter 使用白名单和强类型；
4. phrase 依赖 positions，“倒排结构……索引”不能冒充“倒排 索引”；
5. AND 是 clause 集合交集，OR 是并集；
6. tie-break 固定，重跑得到相同顺序。

更严格的系统还要支持 deny 优先、用户/组/组织层级、文档级与段落级 ACL、生效时间、地域、legal hold，以及撤权后的缓存失效 SLA。

In [ ]:
employee_phrase = search(  # 计算并保存当前步骤的中间状态。
    index, '"倒排 索引"', tenant="acme", roles={"employee"}, top_k=10  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert [hit.doc.doc_id for hit in employee_phrase] == ["search-guide"]  # 用受控断言验证关键不变量。
assert "index-maintenance" not in {hit.doc.doc_id for hit in employee_phrase}  # 用受控断言验证关键不变量。

support_error = search(  # 计算并保存当前步骤的中间状态。
    index, "ERR_AUTH_403", tenant="acme", roles={"support"}, top_k=10  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert [hit.doc.doc_id for hit in support_error] == ["auth-runbook"]  # 用受控断言验证关键不变量。

employee_error = search(  # 计算并保存当前步骤的中间状态。
    index, "ERR_AUTH_403", tenant="acme", roles={"employee"}, top_k=10  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert employee_error == []  # 没有 support/admin 角色，看不到 runbook。

cross_tenant = search(  # 计算并保存当前步骤的中间状态。
    index, "999", tenant="acme", roles={"employee"}, top_k=10  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert cross_tenant == []  # 用受控断言验证关键不变量。

filtered = search(  # 计算并保存当前步骤的中间状态。
    index, "索引", tenant="acme", roles={"employee"}, top_k=10,  # 计算并保存当前步骤的中间状态。
    metadata_filter={"category": "operations"},  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert [hit.doc.doc_id for hit in filtered] == ["index-maintenance"]  # 用受控断言验证关键不变量。
print("ACL、metadata、phrase、字段查询检查全部通过")  # 执行当前语句以推进本节示例。

## 9. 增量 upsert/delete：正确性比“能插入”更难

doc_id 幂等 upsert 的正确顺序是：

1. 校验 schema、权限 metadata、analyzer 版本和版本单调性；
2. 写 WAL/事务日志；
3. 移除旧版本 term 的 postings，再写新版本；
4. 发布新的只读 snapshot/generation；
5. 使查询缓存、结果缓存和高亮缓存按 generation 失效；
6. delete 同时覆盖倒排、正排、向量索引、对象存储副本和合规链路。

本内存实现同步原子完成单进程更新；生产引擎常通过不可变 segment + tombstone + refresh 暴露近实时读。refresh、flush、commit、replica ack 是不同概念，需要明确“写成功后多久可搜到”和“故障后会不会丢”。

In [ ]:
before_generation = index.generation  # 计算并保存当前步骤的中间状态。
new_doc = Document(  # 计算并保存当前步骤的中间状态。
    "cache-runbook", "搜索缓存",  # 执行当前语句以推进本节示例。
    "查询缓存键必须包含 tenant、roles、filter、analyzer_version 和 index_generation。",  # 执行当前语句以推进本节示例。
    "acme", frozenset({"employee"}), "runbook", "active", 1,  # 执行当前语句以推进本节示例。
    ("缓存", "检索"),  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
index.upsert(new_doc)  # 执行当前语句以推进本节示例。
assert index.generation == before_generation + 1  # 用受控断言验证关键不变量。
assert [hit.doc.doc_id for hit in search(  # 用受控断言验证关键不变量。
    index, "缓存", "acme", {"employee"}, top_k=5  # 计算并保存当前步骤的中间状态。
)] == ["cache-runbook"]  # 计算并保存当前步骤的中间状态。

updated_doc = Document(  # 计算并保存当前步骤的中间状态。
    "cache-runbook", "搜索缓存失效",  # 执行当前语句以推进本节示例。
    "缓存键包含租户、角色、过滤器与索引代次；撤权后必须主动失效。",  # 执行当前语句以推进本节示例。
    "acme", frozenset({"admin"}), "runbook", "active", 2,  # 执行当前语句以推进本节示例。
    ("缓存", "权限"),  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
index.upsert(updated_doc)  # 执行当前语句以推进本节示例。
assert "index_generation" not in index.postings["body"]  # 旧 body 的 term 已移除。
assert search(index, "缓存", "acme", {"employee"}, top_k=5) == []  # 用受控断言验证关键不变量。
assert search(index, "缓存", "acme", {"admin"}, top_k=5)[0].doc.version == 2  # 用受控断言验证关键不变量。

assert index.delete("cache-runbook")  # 用受控断言验证关键不变量。
assert not index.delete("cache-runbook")  # 幂等 delete
assert "cache-runbook" not in index.docs  # 用受控断言验证关键不变量。
print("upsert/delete 与权限变更测试通过，generation:", index.generation)  # 执行当前语句以推进本节示例。

## 10. 评估：Recall、MRR、nDCG 各回答不同问题

给查询 q 的相关集合 R 与前 k 个结果 $S_k$：

$$
Recall@k=\frac{|R\cap S_k|}{|R|}.
$$

MRR 只看第一个相关结果：

$$
RR(q)=\frac{1}{rank_{first}},\qquad MRR=\frac{1}{|Q|}\sum_q RR(q).
$$

有多级相关性 $rel_i$ 时：

$$
DCG@k=\sum_{i=1}^{k}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad
nDCG@k=\frac{DCG@k}{IDCG@k}.
$$

Recall 适合候选召回；MRR 适合“第一个答案就够”的导航查询；nDCG 评价整段排序且支持 0/1/2/3 等相关等级。无 gold 的查询不能把 Recall 定义成 1；应单列零结果/拒答、误召回与人工抽检。离线指标还需配合线上成功率、无结果率、点击/解决率、P95/P99 和权限安全指标。

In [ ]:
def recall_at_k(ranked_ids: list[str], relevant: set[str], k: int) -> float:  # 定义本节可复用的核心函数。
    if not relevant:  # 按当前条件选择后续控制路径。
        raise ValueError("无 gold 查询的 Recall@k 未定义")  # 遇到非法合同立即显式失败。
    return len(set(ranked_ids[:k]) & relevant) / len(relevant)  # 返回当前分支计算出的结果。

def reciprocal_rank(ranked_ids: list[str], relevant: set[str]) -> float:  # 定义本节可复用的核心函数。
    return next(  # 返回当前分支计算出的结果。
        (1.0 / rank for rank, doc_id in enumerate(ranked_ids, start=1)  # 计算并保存当前步骤的中间状态。
         if doc_id in relevant),  # 按当前条件选择后续控制路径。
        0.0,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。

def dcg_at_k(ranked_ids: list[str], relevance: dict[str, int], k: int) -> float:  # 定义本节可复用的核心函数。
    return sum(  # 返回当前分支计算出的结果。
        (2 ** relevance.get(doc_id, 0) - 1) / math.log2(rank + 1)  # 执行当前语句以推进本节示例。
        for rank, doc_id in enumerate(ranked_ids[:k], start=1)  # 遍历输入元素以累积或检查结果。
    )  # 执行当前语句以推进本节示例。

def ndcg_at_k(ranked_ids: list[str], relevance: dict[str, int], k: int) -> float:  # 定义本节可复用的核心函数。
    ideal = [  # 计算并保存当前步骤的中间状态。
        doc_id for doc_id, _ in  # 执行当前语句以推进本节示例。
        sorted(relevance.items(), key=lambda item: (-item[1], item[0]))  # 计算并保存当前步骤的中间状态。
    ]  # 执行当前语句以推进本节示例。
    denominator = dcg_at_k(ideal, relevance, k)  # 计算并保存当前步骤的中间状态。
    return dcg_at_k(ranked_ids, relevance, k) / denominator if denominator else 0.0  # 返回当前分支计算出的结果。

eval_cases = [  # 计算并保存当前步骤的中间状态。
    ("BM25 错误码", {"employee", "support"}, {"search-guide": 3}),  # 执行当前语句以推进本节示例。
    ("ERR_AUTH_403", {"support"}, {"auth-runbook": 3}),  # 执行当前语句以推进本节示例。
    ("中文 分词 AX-100", {"employee", "search"}, {"zh-analyzer": 3}),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
rrs, recalls, ndcgs = [], [], []  # 计算并保存当前步骤的中间状态。
for query, roles, qrels in eval_cases:  # 遍历输入元素以累积或检查结果。
    ranked = [  # 计算并保存当前步骤的中间状态。
        hit.doc.doc_id for hit in  # 执行当前语句以推进本节示例。
        search(index, query, "acme", roles, top_k=5)  # 计算并保存当前步骤的中间状态。
    ]  # 执行当前语句以推进本节示例。
    relevant = set(qrels)  # 计算并保存当前步骤的中间状态。
    recalls.append(recall_at_k(ranked, relevant, 3))  # 执行当前语句以推进本节示例。
    rrs.append(reciprocal_rank(ranked, relevant))  # 执行当前语句以推进本节示例。
    ndcgs.append(ndcg_at_k(ranked, qrels, 3))  # 执行当前语句以推进本节示例。
    print(query, "=>", ranked,  # 计算并保存当前步骤的中间状态。
          f"Recall@3={recalls[-1]:.3f}",  # 计算并保存当前步骤的中间状态。
          f"RR={rrs[-1]:.3f}", f"nDCG@3={ndcgs[-1]:.3f}")  # 计算并保存当前步骤的中间状态。

print("macro Recall@3:", sum(recalls) / len(recalls))  # 执行当前语句以推进本节示例。
print("MRR:", sum(rrs) / len(rrs))  # 执行当前语句以推进本节示例。
assert min(recalls) == 1.0  # 用受控断言验证关键不变量。
assert min(rrs) > 0.0  # 用受控断言验证关键不变量。

## 11. top-k、缓存、分片与性能预算

全量遍历并排序是 $O(N\log N)$；倒排查询先取 postings，只对命中文档打分。维护大小为 k 的最小堆，可把候选排序降为 $O(C\log k)$，C 是候选数。真实引擎还会使用 skip list、block-max WAND、impact ordering 提前跳过不可能进入 top-k 的文档。

工程清单：

- **top-k**：服务端限制上限；深分页使用 search_after/游标，不用巨大 offset；
- **缓存**：key 至少含 tenant、主体/角色版本、规范化查询、filter、排序、top_k、索引 generation、analyzer 版本；撤权要主动失效；
- **分片**：优先让 tenant 或业务域有明确路由；每片先取 local top-k，协调节点合并时要留 oversampling；
- **统计量**：分片局部 IDF 会造成排序漂移；可用全局统计、合理路由或接受近似并回归；
- **写入**：批量构建、refresh 周期、segment 数、merge IO 与查询延迟相互影响；
- **观测**：记录解析耗时、filter 后候选数、postings 访问数、打分数、cache hit、各分片耗时和超时降级；
- **容量**：估算 term 数、postings 数、positions/offsets、doc values、replica 与 merge 临时空间。

性能优化前先固定结果正确性；一个很快但越权或漏召回的搜索器没有价值。

In [ ]:
def heap_top_k(scored_doc_ids: Iterable[tuple[float, str]], k: int) -> list[tuple[float, str]]:  # 定义本节可复用的核心函数。
    """用大小为 k 的堆选取结果，并保持分数降序、doc_id 升序。"""  # 执行当前语句以推进本节示例。
    return heapq.nsmallest(k, scored_doc_ids, key=lambda item: (-item[0], item[1]))  # 返回当前分支计算出的结果。

def safe_cache_key(  # 定义本节可复用的核心函数。
    query: str, tenant: str, roles: set[str],  # 执行当前语句以推进本节示例。
    filters: dict[str, object], top_k: int, idx: InMemoryBM25Index,  # 执行当前语句以推进本节示例。
) -> tuple:  # 执行当前语句以推进本节示例。
    return (  # 返回当前分支计算出的结果。
        tenant, tuple(sorted(roles)), query.strip().lower(),  # 执行当前语句以推进本节示例。
        tuple(sorted(filters.items())), top_k,  # 执行当前语句以推进本节示例。
        idx.generation, ANALYZER_VERSION,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。

raw_scores = [(0.5, "b"), (1.2, "c"), (1.2, "a"), (0.7, "d")]  # 计算并保存当前步骤的中间状态。
assert heap_top_k(raw_scores, 3) == [(1.2, "a"), (1.2, "c"), (0.7, "d")]  # 用受控断言验证关键不变量。

key_employee = safe_cache_key("BM25", "acme", {"employee"}, {}, 10, index)  # 计算并保存当前步骤的中间状态。
key_support = safe_cache_key("BM25", "acme", {"support"}, {}, 10, index)  # 计算并保存当前步骤的中间状态。
assert key_employee != key_support  # 用受控断言验证关键不变量。
print("安全缓存键:", key_employee)  # 执行当前语句以推进本节示例。

## 12. 测试策略：把 analyzer、权限和排序当成同等重要的逻辑

至少建立四层测试：

1. **单元测试**：规范化、token/position、DF/TF、IDF、长度归一化、phrase、parser 错误；
2. **属性/不变量**：delete 后无残留 posting；同 doc_id 连续 upsert 不重复；top-k 是全量排序前缀；不可见文档永不返回；
3. **黄金回归集**：查询、身份、过滤器、qrels、预期禁出文档、关键 explain 特征；
4. **影子与线上实验**：新旧索引并跑，比较 Recall/nDCG、零结果率、延迟、资源、越权告警；小流量切换并可回滚。

中文 analyzer 回归集要覆盖全半角、简繁体、emoji、混合大小写、型号、错误码、日期、金额、同义词、停用词与恶意超长输入。安全测试必须包含跨租户、撤权、缓存复用、未知 filter、archived 文档和 explain 权限。

In [ ]:
def run_regression_suite() -> None:  # 定义本节可复用的核心函数。
    # TF 与 positions
    assert index.postings["title"]["bm25"]["search-guide"] == (0,)  # 用受控断言验证关键不变量。
    assert index.doc_frequency("body", "索引", set(index.docs)) >= 2  # 用受控断言验证关键不变量。

    # OR 比 AND 不更窄是常见的不变量（相同身份与过滤条件）。
    roles = {"employee", "support"}  # 计算并保存当前步骤的中间状态。
    or_ids = {h.doc.doc_id for h in search(index, "BM25 错误码", "acme", roles)}  # 计算并保存当前步骤的中间状态。
    and_ids = {h.doc.doc_id for h in search(index, "BM25 AND 错误码", "acme", roles)}  # 计算并保存当前步骤的中间状态。
    assert and_ids <= or_ids  # 用受控断言验证关键不变量。
    assert and_ids == {"search-guide"}  # 用受控断言验证关键不变量。

    # archived 与跨租户绝不出现。
    broad = search(index, "搜索 OR 价格 OR TF-IDF", "acme", roles, top_k=100)  # 计算并保存当前步骤的中间状态。
    broad_ids = {hit.doc.doc_id for hit in broad}  # 计算并保存当前步骤的中间状态。
    assert "search-guide-old" not in broad_ids  # 用受控断言验证关键不变量。
    assert "beta-secret" not in broad_ids  # 用受控断言验证关键不变量。

    # 非法字段与非法 top-k 失败得明确。
    try:  # 尝试执行可能失败的受控操作。
        search(index, "tenant:acme", "acme", roles)  # 执行当前语句以推进本节示例。
        raise AssertionError("未知字段应报错")  # 遇到非法合同立即显式失败。
    except ValueError as error:  # 捕获预期异常并验证失败分支。
        assert "未知全文字段" in str(error)  # 用受控断言验证关键不变量。

    try:  # 尝试执行可能失败的受控操作。
        search(index, "BM25", "acme", roles, top_k=0)  # 计算并保存当前步骤的中间状态。
        raise AssertionError("非法 top_k 应报错")  # 遇到非法合同立即显式失败。
    except ValueError as error:  # 捕获预期异常并验证失败分支。
        assert "top_k" in str(error)  # 用受控断言验证关键不变量。

run_regression_suite()  # 执行当前语句以推进本节示例。
print("最终回归测试全部通过")  # 执行当前语句以推进本节示例。

## 13. 生产落地清单、常见误区与参考资料

### 上线前清单

- schema/analyzer 有版本号、样例 token 与重建方案；
- ACL/tenant/status 在召回前过滤，缓存按身份和 generation 隔离；
- upsert/delete、refresh 可见性与失败重试语义明确；
- explain 可追到 analyzer、TF/DF/IDF、字段与规则 boost；
- qrels 按查询类型、语言、租户与长尾切片，报告 Recall/MRR/nDCG；
- 有性能预算、容量估算、分片路由、超时降级、影子验证与一键回滚。

### 常见误区

1. **BM25 就是词频越高越好**：TF 会饱和，长度、DF 与字段都会影响。
2. **中文逐字切分一定够用**：召回可能高但噪声、phrase 和可解释性会恶化。
3. **先搜全库再做 ACL**：会损失可见 top-k，并制造泄漏面。
4. **字段 boost 就是 BM25F**：逐字段加权和跨字段 TF 组合并不等价。
5. **离线 nDCG 上升即可上线**：还要检查越权、零结果、延迟、成本与切片退化。

### Primary source / 官方文档

- Robertson & Zaragoza, **The Probabilistic Relevance Framework: BM25 and Beyond** (2009)：https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf
- Apache Lucene **BM25Similarity API**（k1、b、idf、field length 的官方实现接口）：https://lucene.apache.org/core/9_9_1/core/org/apache/lucene/search/similarities/BM25Similarity.html
- Elasticsearch **Similarity / BM25** 官方文档：https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/similarity
- Elasticsearch **combined_fields** 官方文档（简化 BM25F 与近似边界）：https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-combined-fields-query
- Elasticsearch **multi_match 与字段 boost** 官方文档：https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-multi-match-query

### 练习

1. 为 postings 增加 character offsets，实现安全高亮。
2. 实现 NOT、括号和最小 should match，并为语法错误返回位置。
3. 将逐字段加权改成 BM25F，并用固定 qrels 比较。
4. 模拟两个分片，比较局部 IDF 与全局 IDF 的排名差异。
5. 将内存更新改成不可变 segment + tombstone + snapshot reader。